# Init Lakehouse

In [12]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [13]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number, desc, trim, regexp_replace
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 3, Finished, Available, Finished)

# Init Export Process

In [14]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path_1(str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [15]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 0

if debug:
    print(debug , incremental_run)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [16]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 6, Finished, Available, Finished)

In [17]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get Issues_Combined from: {filterdate}')

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 7, Finished, Available, Finished)

Get Issues_Combined from: 2025-05-07


# Init Query(s)

In [18]:
issues_dpd_df = spark.sql(f"""
SELECT DISTINCT
	'ENDCTG' AS Company_Code
	,'' AS Site_Code
	,MIN(whscontainertable.shipcarriertrackingnum) AS Issue_Reference
	,CAST(whscontainertable.closecontainerutcdatetime AS DATE) AS Issue_Date
	,'DPDSHIP' AS Project_Reference
	,'' AS IPR_Reference
	,'' AS Stock_Detail_Key_Type
	,'' AS Stock_Detail_Key
	,whscontainerline.itemid AS Product_Part_No_Ref
	,CAST(SUM(whscontainerline.qty) AS DECIMAL(10,2)) AS Quantity
	,'' AS Price
	,'' AS Currency
	,salestable.salesname AS Customer
	,whscontainertable.containerid --- ONLY USED FOR RECORD TRACKING, REMOVE PRIOR TO FILE GEN

FROM whsshipmenttable

INNER JOIN whscontainertable 
	ON whsshipmenttable.shipmentid = whscontainertable.shipmentid
      AND whsshipmenttable.dataareaid = whscontainertable.dataareaid

INNER JOIN whscontainerline
      ON whscontainertable.containerid = whscontainerline.containerid
      AND whscontainertable.dataareaid = whsshipmenttable.dataareaid

INNER JOIN salestable
      ON whsshipmenttable.ordernum = salestable.salesid
      AND whsshipmenttable.dataareaid = salestable.dataareaid

INNER JOIN logisticspostaladdress
      ON salestable.deliverypostaladdress = logisticspostaladdress.recid
	
WHERE
    salestable.dlvmode LIKE '%DPD%'
AND
    logisticspostaladdress.countryregionid != 'GBR'
AND
	whscontainertable.closecontainerutcdatetime >= '{filterdate}'
AND   
    whscontainertable.containerstatus = 2

GROUP BY
	 whscontainertable.shipcarriertrackingnum
	,CAST(whscontainertable.closecontainerutcdatetime AS DATE)
	,whscontainerline.itemid
	,salestable.salesname
	,whscontainertable.containerid
"""
)
if debug:
      display(issues_dpd_df)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9c269861-c75b-47f2-ac93-3f3a926de3ff)

In [19]:
issues_dpd_df = issues_dpd_df.select(
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Site_Code").cast("string"),1, 4).alias("Site_Code"),
    substring(col("Issue_Reference").cast("string"),1, 14).alias("Issue_Reference"), #1, 14 instead of 1, 15 to knock-off the trailing comma
    col("Issue_Date").cast("date").alias("Issue_Date"),
    substring(col("Project_Reference").cast("string"),1, 10).alias("Project_Reference"),
    col("IPR_Reference").cast("string").alias("IPR_Reference"),
    col("Stock_Detail_Key_Type").cast("string").alias("Stock_Detail_Key_Type"),
    col("Stock_Detail_Key").cast("string").alias("Stock_Detail_Key"),
    substring(col("Product_Part_No_Ref").cast("string"),1, 25).alias("Product_Part_No_Ref"),
    substring(col("Quantity").cast("string"),1, 11).alias("Quantity"),
    col("Price").cast("string").alias("Price"),
    col("Currency").cast("string").alias("Currency"),
    col("Customer").cast("string").alias("Customer"),
    col("containerid").cast("string").alias("containerid")
)
if debug:
    display(issues_dpd_df)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 05c2eab1-f505-4a90-a9a5-8323b85802d1)

In [20]:
issues_stock_adjustments_df = spark.sql(f"""
SELECT
	'ENDCTG' AS Company_Code
	,'' AS Site_Code
	,MIN(inventjournaltrans.inventtransid) AS Issue_Reference
	,inventjournaltrans.transdate AS Issue_Date
	,"DESTRUCT" AS Project_Reference
	--,inventjournaltable.journalnameid AS Project_Reference
	,'' AS IPR_Reference
	,'' AS Stock_Detail_Key_Type
	,'' AS Stock_Detail_Key
	,inventjournaltrans.itemid AS Product_Part_No_Ref
	,CAST(SUM(ABS(inventjournaltrans.unitqty)) AS DECIMAL(10,2)) AS Quantity
	,'' AS Price
	,'' AS Currency
	,'' AS Customer

FROM inventjournaltable

INNER JOIN inventjournaltrans 
	ON inventjournaltable.journalid = inventjournaltrans.journalid
    AND inventjournaltable.dataareaid = inventjournaltrans.dataareaid
    AND inventjournaltable.posted  = 1

LEFT JOIN inventtable
    ON inventjournaltrans.itemid = inventtable.itemid
    AND inventjournaltrans.dataareaid = inventtable.dataareaid

LEFT JOIN ecoresproduct
    ON inventtable.product = ecoresproduct.recid

WHERE
	inventjournaltable.journalnameid = 'BWDestroy'
AND
	inventjournaltable.inventlocationid = 'PAR'
AND
	InventJournalTrans.unitqty < 0
AND
	inventjournaltable.dataareaid IN ('end.', 'END.')

AND 
	GREATEST(
		ecoresproduct.modifieddatetime,  
		inventtable.modifieddatetime, 
		inventjournaltrans.modifieddatetime,
		inventjournaltable.modifieddatetime
		) >= '{filterdate}'

GROUP BY
	 inventjournaltrans.inventtransid
	,inventjournaltrans.transdate
	,inventjournaltable.journalnameid
	,inventjournaltrans.itemid
"""
)
if debug:
      display(issues_stock_adjustments_df)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7984edaf-2815-486d-86df-cbc18327ca82)

In [21]:
issues_stock_adjustments_df = issues_stock_adjustments_df.select(
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Site_Code").cast("string"),1, 4).alias("Site_Code"),
    substring(col("Issue_Reference").cast("string"),1, 14).alias("Issue_Reference"), #1, 14 instead of 1, 15 to knock-off the trailing comma
    col("Issue_Date").cast("date").alias("Issue_Date"),
    substring(col("Project_Reference").cast("string"),1, 10).alias("Project_Reference"),
    col("IPR_Reference").cast("string").alias("IPR_Reference"),
    col("Stock_Detail_Key_Type").cast("string").alias("Stock_Detail_Key_Type"),
    col("Stock_Detail_Key").cast("string").alias("Stock_Detail_Key"),
    substring(col("Product_Part_No_Ref").cast("string"),1, 25).alias("Product_Part_No_Ref"),
    substring(col("Quantity").cast("string"),1, 11).alias("Quantity"),
    col("Price").cast("string").alias("Price"),
    col("Currency").cast("string").alias("Currency"),
    col("Customer").cast("string").alias("Customer")
)
if debug:
    display(issues_stock_adjustments_df)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 935af659-6c54-4351-8f93-d332498995ca)

# N/A to NA

In [22]:
issues_dpd_df = issues_dpd_df.replace("N/A", "NA")
issues_stock_adjustments_df = issues_stock_adjustments_df.replace("N/A", "NA")

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 12, Finished, Available, Finished)

## Date Field Changing Post Query(s)

In [23]:
list_date_columns_1 = [name for name, dtype in issues_dpd_df.dtypes if dtype in ('date','timestamp')]
list_date_columns_2 = [name for name, dtype in issues_stock_adjustments_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)
    print("Date Columns to change: " , list_date_columns_2)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 13, Finished, Available, Finished)

Date Columns to change:  ['Issue_Date']
Date Columns to change:  ['Issue_Date']


In [24]:
for column in list_date_columns_1:
    issues_dpd_df = issues_dpd_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

for column in list_date_columns_2:
    issues_stock_adjustments_df = issues_stock_adjustments_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 14, Finished, Available, Finished)

## Display Pre-Filtering

In [25]:
if debug:
    display(issues_dpd_df)
    display(issues_stock_adjustments_df)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, bd5e84fb-199e-4446-813f-53c1f6071a93)

SynapseWidget(Synapse.DataFrame, e5bb5023-9781-42d4-8b14-029890c48c37)

# Init Error Check Process

In [26]:
issues_dpd_Mandatory_Columns = [
    "Company_Code",
    "Issue_Reference",
    "Issue_Date",
    "Project_Reference",
    "Product_Part_No_Ref",
    "Quantity"
]

issues_dpd_Cant_Be_Zero_Columns = [
    "Quantity"
]

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 16, Finished, Available, Finished)

In [27]:
level = 1

null_condition = None
null_column_names_exprs = []

zero_condition = None
zero_column_names_exprs = []


for column in issues_dpd_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

for column in issues_dpd_Cant_Be_Zero_Columns:
    if column == "Quantity":
        condition = col(column) == "0.00"
    zero_condition = condition if zero_condition is None else zero_condition | condition
    zero_column_names_exprs.append(when(condition, lit(column)))


# Collect error columns into arrays
issues_dpd_with_errors = issues_dpd_df.withColumn("null_failed_columns", array(*null_column_names_exprs)) \
                                      .withColumn("zero_failed_columns", array(*zero_column_names_exprs))

# Filter out nulls from those arrays
issues_dpd_with_errors = issues_dpd_with_errors.withColumn(
    "null_failed_columns", expr("filter(null_failed_columns, x -> x is not null)")
).withColumn(
    "zero_failed_columns", expr("filter(zero_failed_columns, x -> x is not null)")
)

# Generate the error messages (only when columns exist)
issues_dpd_with_errors = issues_dpd_with_errors.withColumn(
    "null_errors",
    when(size(col("null_failed_columns")) > 0,
         concat_ws("", lit("Columns "), concat_ws(" , ", col("null_failed_columns")), lit(f" are null at level {level}")))
).withColumn(
    "zero_errors",
    when(size(col("zero_failed_columns")) > 0,
         concat_ws("", concat_ws(", ", col("zero_failed_columns")), lit(f" are 0 at level {level}")))
)

# Combine all errors
issues_dpd_with_errors = issues_dpd_with_errors.withColumn(
    "error_fields",
    concat_ws(" , ", col("null_errors"), col("zero_errors"))
)

# Filter bad and good
issues_dpd_bad_df = issues_dpd_with_errors.filter(null_condition | zero_condition) \
    .drop("null_errors", "zero_errors", "null_failed_columns", "zero_failed_columns","containerid")

issues_dpd_df = issues_dpd_with_errors.filter(~(null_condition | zero_condition)) \
    .drop("error_fields", "null_errors", "zero_errors", "null_failed_columns", "zero_failed_columns")

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 17, Finished, Available, Finished)

In [28]:
if debug:
    display(issues_dpd_bad_df)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 530091c7-2e71-433e-bce2-d392884b691f)

In [29]:
issues_dpd_bad_keys = [row["Issue_Reference"] for row in issues_dpd_bad_df.select("Issue_Reference").distinct().collect()]

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 19, Finished, Available, Finished)

In [30]:
if debug:
    print("Level 1 Bad: " , issues_dpd_bad_keys)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 20, Finished, Available, Finished)

Level 1 Bad:  []


## If it's in Level_1 bad, then is it in in Level_1 good (it should NOT be)

In [31]:
if debug:
    display(issues_dpd_bad_df)
    display(issues_dpd_df)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, e0306ddb-db0b-4fdf-981b-eb15998da398)

SynapseWidget(Synapse.DataFrame, a83c5542-b8a1-48e8-98b2-53bed5a22065)

# Init Good File Name & Date Logic

In [32]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

if now > cutoff_time:
    tomorrow = now + timedelta(days=1)
    #file_datetime = tomorrow.strftime('%Y-%m-%d')
    file_datetime = tomorrow.strftime('%Y-%m-%d-%H')
else:
    #file_datetime = now.strftime('%Y-%m-%d')
    file_datetime = now.strftime('%Y-%m-%d-%H')


file_name_1 = "issues_dpd" + "_" + file_datetime + file_extention
file_path_1 = file_path_folder + file_name_1

file_name_2 = "issues_stock_adjustments" + "_" + file_datetime + file_extention
file_path_2 = file_path_folder + file_name_2

file_name_3 = "issues_combined" + "_" + file_datetime + file_extention
file_path_3 = file_path_folder + file_name_3

file_name_tmp = "issues_combined" + "_tmp" + file_datetime + file_extention
file_path_tmp = file_path_folder + file_name_tmp

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 22, Finished, Available, Finished)

In [33]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name_1)
    print("Good File Path: " , file_path_1)
    print("Good File Name 2: " , file_name_2)
    print("Good File Path 2: " , file_path_2)
    print("Good File Name 3: " , file_name_3)
    print("Good File Path 3: " , file_path_3)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 23, Finished, Available, Finished)

Now:  2025-05-14 11:33:41.761895
Cutoff:  2025-05-14 17:15:00
File Date:  2025-05-14-11
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  issues_dpd_2025-05-14-11.dat
Good File Path:  /lakehouse/default/Files/Output/issues_dpd_2025-05-14-11.dat
Good File Name 2:  issues_stock_adjustments_2025-05-14-11.dat
Good File Path 2:  /lakehouse/default/Files/Output/issues_stock_adjustments_2025-05-14-11.dat
Good File Name 3:  issues_combined_2025-05-14-11.dat
Good File Path 3:  /lakehouse/default/Files/Output/issues_combined_2025-05-14-11.dat


# Remove Rows Already Sent

In [34]:
if incremental_run:
    
    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)

    lakehouse_table_name = "bondedwarehouserecordtracking_issues_dpd"
    container_column = "containerid"

    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()

        # Filter each input dataframe to EXCLUDE already sent
        issues_dpd_df = issues_dpd_df.join(sentrecords_df, issues_dpd_df["containerid"] == sentrecords_df["containerid"], "left_anti")

    except Exception as e:
        print(f"An error occurred: {e}")

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 24, Finished, Available, Finished)

In [35]:
if incremental_run:
    
    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)

    lakehouse_table_name = "bondedwarehouserecordtracking_issues_stock_adjustments"
    container_column = "Issue_Reference"

    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()

        # Filter each input dataframe to EXCLUDE already sent
        issues_stock_adjustments_df = issues_stock_adjustments_df.join(sentrecords_df, issues_stock_adjustments_df["Issue_Reference"] == sentrecords_df["Issue_Reference"], "left_anti")

    except Exception as e:
        print(f"An error occurred: {e}")

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 25, Finished, Available, Finished)

# Export Good

In [36]:
### Create dupe dataframe for record tracking
issues_dpd_df_rt = issues_dpd_df

issues_dpd_df = issues_dpd_df.drop("containerid")

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 26, Finished, Available, Finished)

In [37]:
save_dataframe_to_csv(issues_dpd_df, file_path_1, show_header=False)
save_dataframe_to_csv(issues_stock_adjustments_df, file_path_2, show_header=False)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 27, Finished, Available, Finished)

Add pipes: True
Show headers: False
Add pipes: True
Show headers: False


# Init Bad File Name

In [38]:
error_file = "issues_dpd_errors_" + file_datetime + file_extention
error_file_path = file_path_folder + error_file

if debug:
    print("file: ", error_file, " path: " , error_file_path)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 28, Finished, Available, Finished)

file:  issues_dpd_errors_2025-05-14-11.dat  path:  /lakehouse/default/Files/Output/issues_dpd_errors_2025-05-14-11.dat


# Export Bad

In [39]:
save_dataframe_to_csv(issues_dpd_bad_df, error_file_path, show_header = True)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 29, Finished, Available, Finished)

Add pipes: True
Show headers: True


# Union Good Issues DPD & Issues Stock Adjustments 

In [40]:
if issues_dpd_df.take(1):

    dpd_not_empty = True

else:

    dpd_not_empty = False

if debug:
    print(dpd_not_empty)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 30, Finished, Available, Finished)

True


In [41]:
if issues_dpd_df_rt.take(1):

    dpd_not_empty_rt = True

else:

    dpd_not_empty_rt = False

if debug:
    print(dpd_not_empty_rt)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 31, Finished, Available, Finished)

True


In [42]:
if issues_stock_adjustments_df.take(1):

    sa_not_empty = True

else:

    sa_not_empty = False

if debug:
    print(sa_not_empty)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 32, Finished, Available, Finished)

True


In [43]:
if sa_not_empty:
    if dpd_not_empty:

        dat_file_paths = [
            file_path_1,
            file_path_2
        ]

        if debug:
            list(dat_file_paths)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 33, Finished, Available, Finished)

In [44]:
if sa_not_empty:

    try:
        issues_stock_adjustments_dat = pd.read_csv(file_path_2, sep='|', header=None, dtype=str)
        good_rows = True

    except:
        print("Empty file here at GOOD Issues_SA level")
        good_rows = False

        try:
            for file in dat_file_paths:
                os.remove(file)
                print(f"Deleted: {file}")
        except:
            print("Tried to delete part Issues_SA files, and failed.")

    if debug:
        display(issues_stock_adjustments_dat)


if dpd_not_empty:

    try:
        issues_dpd_dat = pd.read_csv(file_path_1, sep='|', header=None, dtype=str)
        good_rows = True

    except:
        print("Empty file here at GOOD Issues_DPD level")
        good_rows = False

        try:
            for file in dat_file_paths:
                os.remove(file)
                print(f"Deleted: {file}")
        except:
            print("Tried to delete part Issues_DPD files, and failed.")

    if debug:
        display(issues_dpd_dat)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 34, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c3764164-c1ec-40fd-a668-31ac10bb3335)

SynapseWidget(Synapse.DataFrame, 81bd6574-21ac-48d3-819e-3542359af8e9)

In [45]:
if sa_not_empty:
    if dpd_not_empty:
    
        issues_combined = pd.concat([issues_dpd_dat, issues_stock_adjustments_dat], ignore_index=True)
        #issues_dpd_sa = issues_dpd_sa.sort_values(by=[1, 0]).reset_index(drop=True)

        # Convert the DataFrame back to CSV format
        csv_data_before = issues_combined.to_csv(sep='|', index=False, header=False)

        with open(file_path_3, 'w') as file:
            file.write(csv_data_before)
        
        ready_to_copy = True



elif dpd_not_empty:
    issues_not_combined = pd.concat([issues_dpd_dat], ignore_index=True)
    csv_data_no_sa = issues_not_combined.to_csv(sep='|', index=False, header=False)

    with open(file_path_3, 'w') as file:
        file.write(csv_data_no_sa)

    ready_to_copy = True


else:
    ready_to_copy = False

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 35, Finished, Available, Finished)

# Init Record Tracking

In [46]:
if incremental_run:
    if dpd_not_empty_rt:

        # Save the final_df to different tables based on the exportfile variable value

        def record_tracking_df_to_table(dataframe, table_name, file_name_1):
            """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
            table_name_lower = table_name.lower()

            # Check if table exists
            table_exists = True
            try:
                spark.read.table(table_name_lower)
                print(f"Table {table_name_lower} exists.")
            except Exception as e:
                print(f"Table {table_name_lower} does not exist.")
                table_exists = False

            # Columns to deduplicate on (excluding metadata)
            dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

            if table_exists:
                try:
                    dataframe = dataframe.withColumn("ExportName", lit(file_name_1).cast(StringType())) \
                                        .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                    existing_df = spark.read.table(table_name_lower)

                    distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                    initial_existing_count = distinct_existing_df.count()
                    print(f"Existing distinct count: {initial_existing_count}")

                    distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                    combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                        .dropDuplicates(subset=dedup_cols)
                    final_distinct_count = combined_distinct_df.count()
                    print(f"Final distinct count: {final_distinct_count}")

                    rows_added = final_distinct_count - initial_existing_count
                    print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                    combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                    print(f"Distinct records saved to {table_name_lower}.")

                except Exception as e:
                    print(f"Exception: Saving all records in new table. Exception: {e}")
                    dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                    dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                    print(f"Distinct records saved to {table_name_lower}.")

            else:
                try:
                    print(f"Table doesn't exist. Saving all records in new table.")
                    dataframe = dataframe.withColumn("ExportName", lit(file_name_1).cast(StringType())) \
                                        .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                    dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                    dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                    print(f"Distinct records saved to {table_name_lower}.")
                except Exception as e:
                    print(f"Error saving data: {e}")

        table_prefix = 'BondedWarehouseRecordTracking_'

        record_tracking_df_to_table(issues_dpd_df_rt, f"{table_prefix}issues_dpd", file_name_1)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 36, Finished, Available, Finished)

In [47]:
if incremental_run:
    if sa_not_empty:
    
        # Save the final_df to different tables based on the exportfile variable value

        def record_tracking_df_to_table(dataframe, table_name, file_name_2):
            """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
            table_name_lower = table_name.lower()

            # Check if table exists
            table_exists = True
            try:
                spark.read.table(table_name_lower)
                print(f"Table {table_name_lower} exists.")
            except Exception as e:
                print(f"Table {table_name_lower} does not exist.")
                table_exists = False

            # Columns to deduplicate on (excluding metadata)
            dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

            if table_exists:
                try:
                    dataframe = dataframe.withColumn("ExportName", lit(file_name_2).cast(StringType())) \
                                        .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                    existing_df = spark.read.table(table_name_lower)

                    distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                    initial_existing_count = distinct_existing_df.count()
                    print(f"Existing distinct count: {initial_existing_count}")

                    distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                    combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                        .dropDuplicates(subset=dedup_cols)
                    final_distinct_count = combined_distinct_df.count()
                    print(f"Final distinct count: {final_distinct_count}")

                    rows_added = final_distinct_count - initial_existing_count
                    print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                    combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                    print(f"Distinct records saved to {table_name_lower}.")

                except Exception as e:
                    print(f"Exception: Saving all records in new table. Exception: {e}")
                    dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                    dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                    print(f"Distinct records saved to {table_name_lower}.")

            else:
                try:
                    print(f"Table doesn't exist. Saving all records in new table.")
                    dataframe = dataframe.withColumn("ExportName", lit(file_name_2).cast(StringType())) \
                                        .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                    dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                    dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                    print(f"Distinct records saved to {table_name_lower}.")
                except Exception as e:
                    print(f"Error saving data: {e}")

        table_prefix = 'BondedWarehouseRecordTracking_'

        record_tracking_df_to_table(issues_stock_adjustments_df, f"{table_prefix}issues_stock_adjustments", file_name_2)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 37, Finished, Available, Finished)

# Init Send To Azure Blob Storage

In [48]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 38, Finished, Available, Finished)

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [49]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name_3
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name_3

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name_3

if debug:
    print(dest_abfss_file_path)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 39, Finished, Available, Finished)

Files/bonded_warehouse_dev/ToBeSent/issues_combined_2025-05-14-11.dat


In [50]:
source_abfss_file_path = 'Files/Output/' + file_name_3

if debug:
    print(source_abfss_file_path)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 40, Finished, Available, Finished)

Files/Output/issues_combined_2025-05-14-11.dat


In [51]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 41, Finished, Available, Finished)

In [52]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, e284bb8f-98cc-47a1-ac15-ca2d73319b0a, 42, Finished, Available, Finished)

ExitValue: Process Complete